In [1]:
import scanpy as sc
import pandas as pd 

In [2]:
# annotation
anno = pd.read_csv("/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Snapatac2/IPC/atac-meta.csv", index_col=0)
anno

,sample2,barcode2,celltype
AAACAGCCATCGTTCT-1-0,GW6,AAACAGCCATCGTTCT-1,NPC
AAACCGAAGACCATAC-1-0,GW6,AAACCGAAGACCATAC-1,NPC
AAACCGAAGGTCCACA-1-0,GW6,AAACCGAAGGTCCACA-1,NPC
AAACCGAAGTCATGCG-1-0,GW6,AAACCGAAGTCATGCG-1,NPC
AAACCGGCAAATTGCT-1-0,GW6,AAACCGGCAAATTGCT-1,NPC
...,...,...,...
TTTGTGGCAAGTCGCT-1-7,GW20,TTTGTGGCAAGTCGCT-1,IPC
TTTGTGGCACGTAATT-1-7,GW20,TTTGTGGCACGTAATT-1,OPC
TTTGTGGCACTAAATC-1-7,GW20,TTTGTGGCACTAAATC-1,APC
TTTGTGTTCCGCATGA-1-7,GW20,TTTGTGTTCCGCATGA-1,IPC


In [3]:
adata_raw = sc.read_h5ad('/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P2/data/scMulti-omics/1.hscAdata_raw.h5ad')
adata_ref = adata_raw[adata_raw.obs.index.isin(anno.index)]
adata_ref.obs = adata_ref.obs.join(anno)
adata_ref.obs = adata_ref.obs[['sample2', 'barcode2', 'celltype']]
# target_celltypes = [
#     "NPC_proliferative", "NPC", "dIPC", "vIPC", "dorsal_glia", "ventral_glia", "ExN_naive", "ExN_immature_1", "ExN_immature_2", "ExN_immature_3", "ExN_mature_1", "ExN_mature_2", "ExN_mature_3",
#     "InN_naive", "InN_immature_1", "InN_immature_2", "InN_mature_1", "InN_mature_2", "InN_ventral_like", "ventral_unknow", "v0_1", "v2a_b", "MN1", "MN2", "OPC", "Ependymal", "FP", "RP", "Microglia", "VLMC"#, "EC"
# ]
# # 筛选adata对象中指定细胞类型的细胞
# adata_ref = adata_ref[adata_ref.obs['celltype'].isin(target_celltypes)].copy()
adata_ref

AnnData object with n_obs × n_vars = 11745 × 36601
    obs: 'sample2', 'barcode2', 'celltype'
    var: 'gene_ids', 'feature_types'

In [4]:
adata_ref.raw = adata_ref
# sc.pp.normalize_total(adata_ref, target_sum=1e4)
# sc.pp.log1p(adata_ref)
adata_ref.obs = adata_ref.obs.rename(lambda x: f"{x}___cisTopic", axis=0)


In [5]:
adata_ref.obs

,sample2,barcode2,celltype
AAACAGCCATCGTTCT-1-0___cisTopic,GW6,AAACAGCCATCGTTCT-1,NPC
AAACCGAAGACCATAC-1-0___cisTopic,GW6,AAACCGAAGACCATAC-1,NPC
AAACCGAAGGTCCACA-1-0___cisTopic,GW6,AAACCGAAGGTCCACA-1,NPC
AAACCGAAGTCATGCG-1-0___cisTopic,GW6,AAACCGAAGTCATGCG-1,NPC
AAACCGGCAAATTGCT-1-0___cisTopic,GW6,AAACCGGCAAATTGCT-1,NPC
...,...,...,...
TTTGTGGCAAGTCGCT-1-7___cisTopic,GW20,TTTGTGGCAAGTCGCT-1,IPC
TTTGTGGCACGTAATT-1-7___cisTopic,GW20,TTTGTGGCACGTAATT-1,OPC
TTTGTGGCACTAAATC-1-7___cisTopic,GW20,TTTGTGGCACTAAATC-1,APC
TTTGTGTTCCGCATGA-1-7___cisTopic,GW20,TTTGTGTTCCGCATGA-1,IPC


In [6]:
adata_ref.write('/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/IPC/data/ipc.h5ad')

In [7]:
adata_ref.raw.to_adata().obs

,sample2,barcode2,celltype
AAACAGCCATCGTTCT-1-0___cisTopic,GW6,AAACAGCCATCGTTCT-1,NPC
AAACCGAAGACCATAC-1-0___cisTopic,GW6,AAACCGAAGACCATAC-1,NPC
AAACCGAAGGTCCACA-1-0___cisTopic,GW6,AAACCGAAGGTCCACA-1,NPC
AAACCGAAGTCATGCG-1-0___cisTopic,GW6,AAACCGAAGTCATGCG-1,NPC
AAACCGGCAAATTGCT-1-0___cisTopic,GW6,AAACCGGCAAATTGCT-1,NPC
...,...,...,...
TTTGTGGCAAGTCGCT-1-7___cisTopic,GW20,TTTGTGGCAAGTCGCT-1,IPC
TTTGTGGCACGTAATT-1-7___cisTopic,GW20,TTTGTGGCACGTAATT-1,OPC
TTTGTGGCACTAAATC-1-7___cisTopic,GW20,TTTGTGGCACTAAATC-1,APC
TTTGTGTTCCGCATGA-1-7___cisTopic,GW20,TTTGTGTTCCGCATGA-1,IPC


In [ ]:
adata_ref = sc.read_h5ad('/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/IPC/data/ipc.h5ad')

In [8]:
adata = adata_ref

In [9]:
# Saving count data
adata.layers["counts"] = adata.X.copy()
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=10000, batch_key="sample2")
sc.tl.pca(adata)
sc.pp.neighbors(adata, n_neighbors=30, n_pcs=30)
sc.external.pp.bbknn(adata, 
                     batch_key="sample2",
                     n_pcs = 50,
                     use_annoy= False,
                     pynndescent_n_neighbors = 40
                    )  # running bbknn 1.3.6

sc.tl.umap(adata, 
           min_dist = 0.5, 
           spread = 0.8)

/cluster2/huanglab/jiamao/conda/envs/R-4.3.0/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# Obtain cluster-specific differentially expressed genes
sc.tl.rank_genes_groups(adata, groupby="celltype", method="wilcoxon",key_added="rank_genes_celltype")

In [11]:
adata

AnnData object with n_obs × n_vars = 11745 × 36601
    obs: 'sample2', 'barcode2', 'celltype'
    var: 'gene_ids', 'feature_types', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'log1p', 'hvg', 'pca', 'neighbors', 'umap', 'rank_genes_celltype'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [12]:
marker_cell_type = sc.get.rank_genes_groups_df(adata, group=None, key = 'rank_genes_celltype', log2fc_min =0.25, pval_cutoff=0.01)
marker_cell_type

,group,names,scores,logfoldchanges,pvals,pvals_adj
0,APC,SPARCL1,55.110863,2.942390,0.000000,0.000000
1,APC,GLIS3,54.490120,3.345155,0.000000,0.000000
2,APC,TNC,53.612850,3.620454,0.000000,0.000000
3,APC,PRKG1,53.254505,2.919653,0.000000,0.000000
4,APC,ADCY2,52.994694,3.763068,0.000000,0.000000
...,...,...,...,...,...,...
8361,OPC,NPDC1,3.280605,0.739877,0.001036,0.009538
8362,OPC,AL355835.1,3.275937,4.199764,0.001053,0.009685
8363,OPC,MCF2L,3.274218,0.607222,0.001060,0.009733
8364,OPC,AC090825.1,3.273329,0.735760,0.001063,0.009760


In [13]:
marker_cell_type.to_csv('ipc_marker_celltype_de.csv', index=False)